# 07 · Bronze — ingest PDFs with `ai_parse_document`

**Bronze job:** register files, then parse them. Do not extract fields here.

1. `bronze_pdf_files` — binary file registry (`read_files`)
2. `bronze_parsed_pdfs` — `ai_parse_document` VARIANT
3. `bronze_parsed_text` — concatenated page text for Silver

`ai_parse_document` version **2.0**. `descriptionElementTypes` is empty so the call stays cheap on 12 pages.

**Run `05_generate_pet_claim_pdfs` first.**


In [ ]:
%run ./00_setup


In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE bronze_pdf_files
COMMENT 'Binary PDF registry for pet-insurance claim packets'
AS
SELECT
  path,
  modificationTime,
  length,
  content,
  regexp_extract(path, '(CLM-PET-[0-9]+)', 1) AS claim_id,
  element_at(split(path, '/'), -1) AS file_name
FROM read_files('{PDF_DIR}', format => 'binaryFile')
WHERE path LIKE '%.pdf'
""")

display(spark.sql("SELECT claim_id, file_name, length FROM bronze_pdf_files ORDER BY claim_id, file_name"))
print("rows", spark.table("bronze_pdf_files").count())


In [ ]:
spark.sql("""
CREATE OR REPLACE TABLE bronze_parsed_pdfs
COMMENT 'ai_parse_document VARIANT output (v2.0) for pet-claim PDFs'
AS
SELECT
  claim_id,
  file_name,
  path,
  length,
  modificationTime,
  ai_parse_document(
    content,
    map('version', '2.0', 'descriptionElementTypes', '')
  ) AS parsed
FROM bronze_pdf_files
""")

# Do not display the VARIANT column — Spark Connect cannot materialize it in the notebook UI.
display(spark.sql("""
SELECT claim_id, file_name, length, typeof(parsed) AS parsed_type
FROM bronze_parsed_pdfs
ORDER BY claim_id, file_name
"""))


In [ ]:
# Flatten parsed elements → one text blob per file.
# Cast VARIANT arrays through to_json/from_json so transform() sees ARRAY<STRUCT>.
spark.sql("""
CREATE OR REPLACE TABLE bronze_parsed_text
COMMENT 'Flattened parse text, one row per PDF'
AS
SELECT
  claim_id,
  file_name,
  path,
  concat_ws('\n', transform(
    from_json(to_json(parsed:document:elements), 'array<struct<content:string>>'),
    e -> e.content
  )) AS parsed_text
FROM bronze_parsed_pdfs
""")

display(spark.sql("""
SELECT claim_id, file_name, length(parsed_text) AS n_chars, substr(parsed_text, 1, 240) AS preview
FROM bronze_parsed_text
ORDER BY claim_id, file_name
"""))
